# Phase 4: Evaluation Framework

This notebook documents the category-level evaluation framework built in Phase 4.  
Instead of relying on a single aggregate MAE, we break predictions into **decision-relevant categories** and evaluate each model where it matters most.

---
## 1. Why Aggregate Metrics Aren't Enough

A single MAE number can hide serious failures. Consider the **mode collapse** problem we encountered:

- The chain-of-thought LLM strategy achieved an overall MAE of **2.20** -- competitive with XGBoost (2.11).
- But it was predicting only **2 unique values** (0 and 1) across 243 players.
- The low MAE happened because most FPL players score 1-3 points in a given gameweek, so predicting "1" for everyone gets you a decent average error.
- For actual FPL decisions (who to captain, who to bench), this model is **completely useless** -- it cannot distinguish between a premium captain pick and a benchwarmer.

This is why we need **category-level evaluation**. The categories are designed around the decisions FPL managers actually make:

| Decision | What can go wrong | Eval category |
|:---------|:-----------------|:--------------|
| Captain choice | Model predicts 4 for everyone | `high_value: captain_candidates` |
| Bench a rotation risk | Model ignores low minutes | `edge_cases: player_low_minutes` |
| Start home vs away | Model has no fixture awareness | `fixture_context: home_vs_away_all` |
| Avoid cold players | Model still rates out-of-form players | `edge_cases: cold_player` |

Aggregate MAE tells you the model is "OK on average." Category evals tell you **where it breaks**.

---
## 2. The Eval Suite

The eval suite is defined in `eval/eval_suite.yaml`. Each category groups related test scenarios with specific filter criteria applied to the GW31+ test set.

In [ ]:
import yaml
import json
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

with open(os.path.join(PROJECT_ROOT, "eval", "eval_suite.yaml")) as f:
    suite = yaml.safe_load(f)

for cat_name, cat in suite["categories"].items():
    print(f"\n{'='*60}")
    print(f"Category: {cat_name}")
    print(f"  {cat['description']}")
    print(f"  Cases:")
    for case in cat["cases"]:
        print(f"    - {case['name']}: {case['description']}")
        print(f"      Filters: {case.get('filter', {})}")
        print(f"      Expected range: {case.get('expected_range', 'N/A')}")

**What each category tests:**

- **fixture_context** -- Does the model understand that playing at home against a weak team is different from playing away against a top-6 side? This matters for every transfer and captain decision.

- **positional** -- FPL scoring is fundamentally different by position. Goalkeepers get clean sheet points, forwards get goal points. A model that treats them the same will fail.

- **edge_cases** -- The tricky situations: players with low minutes (rotation risk), hot streaks (regression to mean), cold players (should predict low), struggling teams. These are where naive models break.

- **high_value** -- The predictions that actually swing your FPL rank. Captain candidates are worth 2x points. Getting these wrong costs you more than getting a bench player wrong.

---
## 3. Running the Eval

The eval pipeline is a single command that:
1. Builds eval cases from `eval_suite.yaml` by filtering the test set (GW31+)
2. Loads existing prediction files (no re-inference needed)
3. Scores each model against every eval case
4. Saves timestamped results

```bash
python eval/run_eval.py --tag v3-baseline
```

To also check for regressions against a saved baseline:
```bash
python eval/run_eval.py --tag v3-baseline --compare-baseline
```

In [ ]:
# Load the results from the v3-baseline eval run
results_path = os.path.join(
    PROJECT_ROOT, "eval", "results", "2026-03-26_v3-baseline", "scores.json"
)
with open(results_path) as f:
    results = json.load(f)

print(f"Eval tag: {results['tag']}")
print(f"Timestamp: {results['timestamp']}")
print(f"Test size: {results['test_size']} player-gameweeks")
print(f"Models evaluated: {list(results['models'].keys())}")

# Show overall metrics
print(f"\n{'Model':<25} {'MAE':>6} {'RMSE':>6} {'Within 1':>9} {'Within 3':>9}")
print("-" * 60)
for model_name, model_data in results["models"].items():
    o = model_data["overall"]
    print(
        f"{model_name:<25} {o['mae']:>6.3f} {o['rmse']:>6.3f}"
        f" {o['within_1']:>8.1%} {o['within_3']:>8.1%}"
    )

---
## 4. Category Breakdown

The grouped bar chart below shows MAE by category for each model. This is where the story gets interesting -- overall MAE hides huge differences in category-level performance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract category MAEs for each model (exclude zero_shot -- it is an outlier that
# compresses the chart; we discuss it separately)
models_to_plot = ["xgboost", "llm_few_shot", "llm_chain_of_thought", "llm_fine_tuned"]
categories = ["fixture_context", "positional", "edge_cases", "high_value"]
cat_labels = ["Fixture\nContext", "Positional", "Edge\nCases", "High\nValue"]

data = {}
for model in models_to_plot:
    model_data = results["models"][model]
    data[model] = [
        model_data["categories"].get(cat, {}).get("category_mae", 0)
        for cat in categories
    ]

x = np.arange(len(categories))
width = 0.2
colors = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D"]
model_labels = ["XGBoost", "Few-Shot", "Chain-of-Thought", "Fine-Tuned"]

fig, ax = plt.subplots(figsize=(12, 6))
for i, (model, label) in enumerate(zip(models_to_plot, model_labels)):
    offset = (i - len(models_to_plot) / 2 + 0.5) * width
    bars = ax.bar(x + offset, data[model], width, label=label, color=colors[i],
                  edgecolor="white", linewidth=0.5)
    # Add value labels on bars
    for bar, val in zip(bars, data[model]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f"{val:.2f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_ylabel("MAE (lower is better)", fontsize=12)
ax.set_title("Category-Level MAE by Model (v3-baseline)", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(cat_labels, fontsize=11)
ax.legend(loc="upper left", fontsize=10)
ax.set_ylim(0, max(max(v) for v in data.values()) * 1.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

**Key observations:**

- **XGBoost** is the most consistent across categories. It does not have spectacular wins but avoids catastrophic failures.

- **Few-Shot** beats XGBoost on edge cases (1.67 vs 1.83) -- especially cold players (1.42 vs 1.65). The examples in the prompt help the LLM reason about out-of-form players.

- **Chain-of-Thought** has the best raw edge-case MAE (1.56) but this is misleading -- it achieves this by predicting very low values for almost everything (only 2 unique output values). It looks good on cold/low-minutes players because those players actually score low.

- **High Value is the hardest category for every model.** Captain candidates have MAE ranging from 4.28 (XGBoost) to 7.00 (fine-tuned). These are the predictions that matter most and where we have the most room to improve.

- **Fine-Tuned is the worst at high-value predictions** (4.22 category MAE). Despite being trained on FPL data, it over-predicts for premium players. The LoRA fine-tuning seems to have amplified the LLM's tendency to be optimistic about in-form players.

In [ ]:
# Include zero-shot for completeness -- show why it was excluded from the main chart
print("Zero-shot LLM category MAEs (excluded from chart due to scale):")
print()
zs = results["models"]["llm_zero_shot"]
for cat in categories:
    mae = zs["categories"].get(cat, {}).get("category_mae", "N/A")
    print(f"  {cat:<20} MAE: {mae}")
print()
print("The zero-shot model wildly over-predicts (mean prediction ~15 vs actual ~3).")
print("It has no calibration to FPL's scoring scale without examples or fine-tuning.")

---
## 5. Output Quality Checks

Beyond accuracy, we need to verify that LLM outputs are structurally valid. The checks in `eval/checks/output_checks.py` catch:

1. **Parse failures** -- Did the model return a valid integer?
2. **Plausible range** -- Is the prediction between -4 and 25 (the realistic FPL range)?
3. **Expected range** -- Does the prediction fall within the case's expected range?
4. **Distribution check (mode collapse)** -- Are predictions collapsed to a single value?

In [ ]:
# Distribution check results for each LLM strategy
print("Distribution Check (Mode Collapse Detection)")
print("=" * 55)
print()

llm_models = ["llm_zero_shot", "llm_few_shot", "llm_chain_of_thought", "llm_fine_tuned"]

for model_name in llm_models:
    model_data = results["models"][model_name]
    dist = model_data.get("checks", {}).get("distribution", {})
    status = "PASS" if dist.get("pass") else "FAIL"
    unique = dist.get("unique_values", "?")
    values = dist.get("values", [])
    print(f"{model_name:<25} [{status}]  {unique} unique values: {values}")

print()
print("All 4 strategies pass the distribution check (> 1 unique value).")
print()
print("However, chain_of_thought only outputs 2 unique values (0 and 1).")
print("Technically it passes the mode-collapse check, but in practice")
print("this is nearly as useless as full collapse. A stricter threshold")
print("(e.g., requiring >= 5 unique values) would flag this.")

In [ ]:
# Parse success and check pass rates by model and category
print("Per-case check failure rates (LLM models only)")
print("=" * 65)
print()
print(f"{'Model':<25} {'Case':<25} {'Failures':>8} {'Total':>6} {'Rate':>7}")
print("-" * 72)

for model_name in llm_models:
    model_data = results["models"][model_name]
    for cat_name, cat_data in model_data["categories"].items():
        for case_name, case_data in cat_data["cases"].items():
            checks = case_data.get("checks")
            if checks and checks.get("failures", 0) > 0:
                rate = checks["failures"] / checks["total"] * 100
                print(
                    f"{model_name:<25} {case_name:<25}"
                    f" {checks['failures']:>8} {checks['total']:>6} {rate:>6.1f}%"
                )

**Why output quality checks matter:**

An LLM that returns "I think this player will do well" instead of a number will silently break your pipeline. The checks catch this early:

- **Parse check** ensures downstream code always gets an integer.
- **Plausible range** catches hallucinated values like 150 points.
- **Distribution check** catches the subtle failure where everything looks fine on aggregate but the model has learned to output a single safe value.

These checks run automatically as part of `run_eval.py` and are reported per-case in the results JSON.

---
## 6. Individual Case Deep Dives

Let's look at three interesting cases in detail to understand model behavior.

In [ ]:
# Case 1: captain_candidates (high_value) -- the highest-stakes prediction
print("CASE: captain_candidates")
print("Description: Players you'd consider captaining (high price, good form, easy fixture)")
print("Why it matters: Captain gets 2x points. Wrong captain pick = 10+ point swing.")
print()

all_models = ["xgboost", "llm_zero_shot", "llm_few_shot", "llm_chain_of_thought", "llm_fine_tuned"]
case_name = "captain_candidates"
cat_name = "high_value"

print(f"{'Model':<25} {'MAE':>6} {'Mean Pred':>10} {'Mean Actual':>12} {'Within 3':>9}")
print("-" * 65)
for model_name in all_models:
    case = results["models"][model_name]["categories"][cat_name]["cases"][case_name]
    print(
        f"{model_name:<25} {case['mae']:>6.2f}"
        f" {case['mean_prediction']:>10.2f} {case['mean_actual']:>12.2f}"
        f" {case['within_3']:>8.1%}"
    )

print()
print("Insight: Every model struggles here. XGBoost is closest (MAE 4.28) but still")
print("far from useful for captain decisions. Zero-shot massively over-predicts (18.67")
print("mean prediction vs 5.33 actual). Fine-tuned over-predicts too (9.67 mean).")

In [ ]:
# Case 2: cold_player (edge_cases) -- few-shot's biggest win
print("CASE: cold_player")
print("Description: Players with very low recent form (form_3gw <= 1.5)")
print("Why it matters: Holding a cold player costs you a transfer. Need accurate low predictions.")
print()

case_name = "cold_player"
cat_name = "edge_cases"

print(f"{'Model':<25} {'MAE':>6} {'Mean Pred':>10} {'Mean Actual':>12} {'Within 1':>9}")
print("-" * 65)
for model_name in all_models:
    case = results["models"][model_name]["categories"][cat_name]["cases"][case_name]
    print(
        f"{model_name:<25} {case['mae']:>6.2f}"
        f" {case['mean_prediction']:>10.2f} {case['mean_actual']:>12.2f}"
        f" {case['within_1']:>8.1%}"
    )

print()
print("Insight: Few-shot (1.42) and chain-of-thought (1.34) beat XGBoost (1.65).")
print("The LLM with examples understands that low-form players score low.")
print("But chain-of-thought achieves this by predicting ~1 for everything --")
print("it wins here by accident, not by understanding.")

In [ ]:
# Case 3: player_on_hot_streak (edge_cases) -- the regression-to-mean trap
print("CASE: player_on_hot_streak")
print("Description: Players with 7+ form -- on a hot streak")
print("Why it matters: Do you hold/buy a streaking player or expect regression?")
print()

case_name = "player_on_hot_streak"
cat_name = "edge_cases"

print(f"{'Model':<25} {'MAE':>6} {'Mean Pred':>10} {'Mean Actual':>12} {'Count':>6}")
print("-" * 62)
for model_name in all_models:
    case = results["models"][model_name]["categories"][cat_name]["cases"][case_name]
    print(
        f"{model_name:<25} {case['mae']:>6.2f}"
        f" {case['mean_prediction']:>10.2f} {case['mean_actual']:>12.2f}"
        f" {case['count']:>6}"
    )

print()
print("Insight: Small sample (4 players) but telling. XGBoost (3.29) under-predicts")
print("at 4.08 vs actual 5.00 -- it regresses too aggressively. Fine-tuned (6.25)")
print("over-predicts at 9.25 -- it extrapolates the streak too far. Few-shot (3.75)")
print("is closest with 6.25 mean prediction, slightly above actual.")

In [ ]:
# Visual comparison of the three deep-dive cases
deep_dive_cases = [
    ("high_value", "captain_candidates"),
    ("edge_cases", "cold_player"),
    ("edge_cases", "player_on_hot_streak"),
]
plot_models = ["xgboost", "llm_few_shot", "llm_fine_tuned"]
plot_labels = ["XGBoost", "Few-Shot", "Fine-Tuned"]
plot_colors = ["#2E86AB", "#A23B72", "#C73E1D"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (cat, case_name) in zip(axes, deep_dive_cases):
    preds = []
    actuals = []
    for model in plot_models:
        case = results["models"][model]["categories"][cat]["cases"][case_name]
        preds.append(case["mean_prediction"])
        actuals.append(case["mean_actual"])

    x_pos = np.arange(len(plot_models))
    bar_w = 0.35

    bars1 = ax.bar(x_pos - bar_w / 2, preds, bar_w, label="Predicted",
                   color=plot_colors, edgecolor="white", linewidth=0.5)
    bars2 = ax.bar(x_pos + bar_w / 2, actuals, bar_w, label="Actual",
                   color="#888888", edgecolor="white", linewidth=0.5, alpha=0.7)

    ax.set_title(case_name.replace("_", " ").title(), fontsize=11, fontweight="bold")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(plot_labels, fontsize=9, rotation=15)
    ax.set_ylabel("Mean Points")
    ax.legend(fontsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Predictions vs Actuals -- Three Deep-Dive Cases", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 7. Regression Detection

When iterating on models, we need to know if a change made things worse. The regression detector (`eval/compare.py`) compares two eval runs and flags any case where MAE increased beyond a threshold.

### How it works

1. Load the new eval results and a saved baseline (`eval/baseline/scores.json`).
2. For each model, compare MAE at three levels:
   - **Overall** -- did the model get worse globally?
   - **Category** -- did a specific category regress?
   - **Case** -- did a specific eval case regress?
3. Flag any delta exceeding the threshold (default: 0.1 MAE points).

```bash
python eval/compare.py \
    --new eval/results/2026-03-26_v3-baseline/scores.json \
    --baseline eval/baseline/scores.json \
    --threshold 0.1
```

### Example output format

```
WARNING: 3 REGRESSIONS DETECTED
--------------------------------------------------
  llm_fine_tuned / category:high_value
    MAE: 3.800 -> 4.222 (+0.422)
  llm_fine_tuned / case:captain_candidates
    MAE: 6.500 -> 7.000 (+0.500)
  llm_few_shot / case:player_on_hot_streak
    MAE: 3.200 -> 3.750 (+0.550)

2 IMPROVEMENTS:
--------------------------------------------------
  xgboost / case:cold_player
    MAE: 1.800 -> 1.645 (-0.155)
  llm_few_shot / category:edge_cases
    MAE: 1.820 -> 1.674 (-0.146)
```

In [ ]:
# Demonstrate the compare function with the actual code
import sys
sys.path.insert(0, PROJECT_ROOT)
from eval.compare import compare_runs

# Load baseline if it exists
baseline_path = os.path.join(PROJECT_ROOT, "eval", "baseline", "scores.json")

if os.path.exists(baseline_path):
    with open(baseline_path) as f:
        baseline = json.load(f)

    comparison = compare_runs(results, baseline, threshold=0.1)
    print(f"Comparison summary: {comparison['summary']}")
    print()

    if comparison["regressions"]:
        print("REGRESSIONS:")
        for r in comparison["regressions"]:
            print(f"  {r['model']} / {r['level']}")
            print(f"    MAE: {r['baseline_mae']:.3f} -> {r['new_mae']:.3f} ({r['delta']:+.3f})")

    if comparison["improvements"]:
        print("\nIMPROVEMENTS:")
        for i in comparison["improvements"]:
            print(f"  {i['model']} / {i['level']}")
            print(f"    MAE: {i['baseline_mae']:.3f} -> {i['new_mae']:.3f} ({i['delta']:+.3f})")
else:
    print(f"No baseline found at {baseline_path}")
    print("To set the current run as baseline:")
    print(f"  cp {results_path} {baseline_path}")

**Why regression detection matters for iterative development:**

Without it, you can easily "improve" a model on one dimension while silently breaking another. For example:

- Tuning the fine-tuned model to better predict captain candidates might make it worse on cold players.
- A new prompt template for few-shot might improve fixture awareness but lose edge-case handling.

The threshold-based approach (default 0.1 MAE) avoids noise -- small fluctuations are ignored, but meaningful regressions are flagged immediately. This is especially important because we run evals on a relatively small test set (243 player-gameweeks), where individual case counts can be as small as 3.

---
## 8. What Category Evals Revealed

The category-level evaluation surfaced insights that aggregate MAE completely hides:

In [ ]:
# Summary table: key case MAEs for the three competitive models
key_cases = [
    ("edge_cases", "cold_player", "Cold Player"),
    ("edge_cases", "player_low_minutes", "Low Minutes"),
    ("fixture_context", "home_vs_away_all", "Home (fixture)"),
    ("high_value", "captain_candidates", "Captain Picks"),
]
key_models = ["xgboost", "llm_few_shot", "llm_fine_tuned"]
key_labels = ["XGBoost", "Few-Shot", "Fine-Tuned"]

print(f"{'Case':<20}", end="")
for label in key_labels:
    print(f" {label:>12}", end="")
print(f" {'Best':>12}")
print("-" * 60)

for cat, case_name, display_name in key_cases:
    print(f"{display_name:<20}", end="")
    maes = []
    for model in key_models:
        mae = results["models"][model]["categories"][cat]["cases"][case_name]["mae"]
        maes.append(mae)
        print(f" {mae:>12.2f}", end="")
    best_idx = maes.index(min(maes))
    print(f" {key_labels[best_idx]:>12}")

print()

### Key insights

**1. Few-shot beats XGBoost on cold players (1.42 vs 1.65)**

With a few examples of low-form players returning low points, the LLM learns to predict conservatively. XGBoost's feature-based approach tends to over-predict slightly for cold players because it weighs historical base rates.

**2. All models struggle with captain picks**

Captain candidates (MAE 4.28 to 7.00) are the hardest prediction in FPL. These are high-variance players -- they either haul big or blank. Even XGBoost, the best performer, has only 66.7% of predictions within 3 points. This is the most important area to improve.

**3. Fine-tuned is worst at high-value predictions**

Despite being trained on FPL data, the fine-tuned model has the highest MAE for captain candidates (7.00) and budget enablers (4.02). The LoRA adapter seems to have over-fit to average-case predictions and lost the ability to differentiate premium players. This is a clear signal that fine-tuning needs more high-value examples in the training set.

**4. Chain-of-thought's "good" numbers are misleading**

Chain-of-thought has the best edge-case MAE (1.56) but only outputs 2 unique values. It wins on low-scoring categories by accident. The distribution check catches this, but only if you look at it -- the MAE alone would fool you.

**5. No single model wins everywhere**

XGBoost is best at fixture context and captain picks. Few-shot is best at cold players and low-minutes. This suggests an **ensemble approach** -- use different models for different decision types -- could be the path forward.

---
## 9. Building Your Own Eval Suite

If you're adapting this framework for a different prediction problem, here is the approach:

### Start with the decisions, work backward

1. **List the decisions your users make.** For FPL: captain choice, transfer in/out, bench order, chip timing.

2. **For each decision, identify what a wrong prediction looks like.** Captaining a 2-pointer instead of a 10-pointer. Benching a player who hauls. Transferring in a player about to get injured.

3. **Define filters that isolate those scenarios.** Captain candidates are expensive, in-form, with easy fixtures. Bench decisions involve rotation risks with low minutes.

4. **Set expected ranges based on domain knowledge.** A cold player should score 0-4. A captain candidate should score 4-15. These aren't hard constraints -- they're sanity checks.

### Practical tips

- **Keep cases small and focused.** A case with 500 rows tells you nothing more than overall MAE. A case with 3-10 rows for a specific scenario gives you actionable signal.

- **Include at least one "high-value" category.** Not all predictions are equal. A captain prediction is worth 10x a bench prediction. Weight your eval accordingly.

- **Always include output quality checks.** Mode collapse, parse failures, and range violations are easy to miss with aggregate metrics.

- **Set a baseline early.** Even a bad baseline gives you something to compare against. Run `compare.py` on every change.

- **Expect small samples.** Edge cases are edge cases for a reason. You might have 3-4 captain candidates in a test set. That is fine -- the eval is directional, not statistically rigorous.

### The eval suite YAML format

```yaml
categories:
  your_category:
    description: "What this category tests"
    cases:
      - name: case_name
        description: "Human-readable description"
        criteria: "What a good prediction looks like"
        filter:
          column_name: exact_value
          column_name_min: minimum_value
          column_name_max: maximum_value
        expected_range: [low, high]
```

Filters support exact match, `_min` suffix (>=), and `_max` suffix (<=). Cases are built automatically by `eval/build_cases.py` which applies these filters to the test set and writes one JSONL file per case.

In [ ]:
# Show the manifest -- the generated cases with their row counts
manifest_path = os.path.join(PROJECT_ROOT, "eval", "cases", "manifest.json")
with open(manifest_path) as f:
    manifest = json.load(f)

print(f"{'Case':<30} {'Category':<20} {'Count':>6}")
print("-" * 58)
for case in manifest:
    print(f"{case['name']:<30} {case['category']:<20} {case['count']:>6}")
print(f"\nTotal cases: {len(manifest)}")